# 17 · Cross-Dataset Evaluation (ESC-50 Out-of-Domain Generalization)
Evaluate pre-trained candidate acoustic models (SVM-clean, RF-clean, SVM-mixed, RF-mixed) on held-out ESC-50 clips covering shared classes (traffic_horn, construction, siren), quantify cross-dataset performance drop vs in-dataset fold 10, analyze impossible-class confusions, compute bootstrap confidence intervals, and export results.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import accuracy_score, f1_score, classification_report

print('Successfully imported standard libraries, models, and metrics')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported standard libraries, models, and metrics


In [2]:
# Cell 2 – verify sha256 of inputs, load esc_idx, X_esc, y_esc, assert split & class distributions, load models
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'
split_path = REPO_ROOT / 'data/processed/split_indices.npz'
svm_clean_path = REPO_ROOT / 'models/svm_candidate.joblib'
rf_clean_path = REPO_ROOT / 'models/rf_candidate.joblib'
svm_mixed_path = REPO_ROOT / 'models/svm_mixed_candidate.joblib'
rf_mixed_path = REPO_ROOT / 'models/rf_mixed_candidate.joblib'

EXPECTED_HASHES = {
    labels_path: '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854',
    features_path: 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a',
    split_path: '2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b',
    svm_clean_path: '3e798d576f2d30e7bb4ae19345879598527d07f33385d9b9f1529d433de3d9e2',
    rf_clean_path: '40f38b939c23930f621fa3ba66de0ce06c9f2fec8da70aeaf9b87ad8ed6d6113',
    svm_mixed_path: '72543a986a3178f952f7a935ab7054ad44cfc1e52dbcae30a2756fee95dbb03d',
    rf_mixed_path: '7e2ecaefce463234cc71ad43e73a7d9cf0e73ee78e35ccde43317b6ffaa15a56'
}

for path, expected_sha in EXPECTED_HASHES.items():
    actual_sha = compute_sha256(path)
    rel_path = path.relative_to(REPO_ROOT)
    print(f'{str(rel_path):40s}: {actual_sha}')
    assert actual_sha == expected_sha, f'SHA256 mismatch for {rel_path}: expected {expected_sha}, got {actual_sha}'

print('All 7 input SHA256 hashes verified successfully ✓')

# Load labels, audio features, and split indices
labels = pd.read_csv(labels_path)
audio_features = np.load(features_path)

with np.load(split_path) as split_data:
    train_idx = split_data['train_idx']
    test_idx = split_data['test_idx']
    esc_idx = split_data['esc_idx']

X_esc = audio_features[esc_idx]
y_esc = labels.loc[esc_idx, 'app_class'].to_numpy()

print(f'Loaded esc_idx length: {len(esc_idx)}')
print(f'X_esc shape: {X_esc.shape}, y_esc shape: {y_esc.shape}')

# Assertions
assert len(esc_idx) == 200, f'Expected 200 rows in esc_idx, got {len(esc_idx)}'
assert X_esc.shape == (200, 128), f'Expected X_esc shape (200, 128), got {X_esc.shape}'
assert len(y_esc) == 200, f'Expected 200 labels in y_esc, got {len(y_esc)}'

n_horn = int(np.sum(y_esc == 'traffic_horn'))
n_const = int(np.sum(y_esc == 'construction'))
n_siren = int(np.sum(y_esc == 'siren'))
print(f'ESC-50 class counts: traffic_horn={n_horn}, construction={n_const}, siren={n_siren}')

assert n_horn == 80, f'Expected 80 traffic_horn clips, got {n_horn}'
assert n_const == 80, f'Expected 80 construction clips, got {n_const}'
assert n_siren == 40, f'Expected 40 siren clips, got {n_siren}'

overlap_train = set(esc_idx) & set(train_idx)
overlap_test = set(esc_idx) & set(test_idx)
assert len(overlap_train) == 0, f'Detected {len(overlap_train)} overlapping indices between ESC-50 and train!'
assert len(overlap_test) == 0, f'Detected {len(overlap_test)} overlapping indices between ESC-50 and test!'
print('Disjoint split assertion passed: no ESC-50 row is in train or test sets ✓')

# Load models
models = {
    'SVM-clean': joblib.load(svm_clean_path),
    'RF-clean': joblib.load(rf_clean_path),
    'SVM-mixed': joblib.load(svm_mixed_path),
    'RF-mixed': joblib.load(rf_mixed_path)
}
print(f'Loaded {len(models)} models: {list(models.keys())}')
print('Cell 2 asserts passed ✓')

data\processed\labels.csv               : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
data\processed\audio_features.npy       : ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
data\processed\split_indices.npz        : 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
models\svm_candidate.joblib             : 3e798d576f2d30e7bb4ae19345879598527d07f33385d9b9f1529d433de3d9e2
models\rf_candidate.joblib              : 40f38b939c23930f621fa3ba66de0ce06c9f2fec8da70aeaf9b87ad8ed6d6113
models\svm_mixed_candidate.joblib       : 72543a986a3178f952f7a935ab7054ad44cfc1e52dbcae30a2756fee95dbb03d


models\rf_mixed_candidate.joblib        : 7e2ecaefce463234cc71ad43e73a7d9cf0e73ee78e35ccde43317b6ffaa15a56
All 7 input SHA256 hashes verified successfully ✓
Loaded esc_idx length: 200
X_esc shape: (200, 128), y_esc shape: (200,)
ESC-50 class counts: traffic_horn=80, construction=80, siren=40
Disjoint split assertion passed: no ESC-50 row is in train or test sets ✓


Loaded 4 models: ['SVM-clean', 'RF-clean', 'SVM-mixed', 'RF-mixed']
Cell 2 asserts passed ✓


In [3]:
# Cell 3 – evaluate 4 models on ESC-50 (accuracy, macro-F1, per-class metrics, impossible class counts)
three_classes = ['construction', 'siren', 'traffic_horn']
impossible_classes = ['hvac', 'music_loudspeaker']

cross_eval = {}

for model_name, model in models.items():
    y_pred_esc = model.predict(X_esc)
    acc = float(accuracy_score(y_esc, y_pred_esc))
    macro_f1 = float(f1_score(y_esc, y_pred_esc, labels=three_classes, average='macro'))
    rep = classification_report(y_esc, y_pred_esc, labels=three_classes, output_dict=True, zero_division=0)
    
    n_hvac = int(np.sum(y_pred_esc == 'hvac'))
    n_music = int(np.sum(y_pred_esc == 'music_loudspeaker'))
    n_impossible = n_hvac + n_music
    
    cross_eval[model_name] = {
        'y_pred': y_pred_esc,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'per_class': rep,
        'n_hvac': n_hvac,
        'n_music_loudspeaker': n_music,
        'n_impossible': n_impossible
    }
    
    print(f'=== {model_name} on ESC-50 (Cross-Dataset, 200 clips) ===')
    print(f'Accuracy (200 clips) : {acc:.4f} ({acc*100:.2f}%)')
    print(f'Macro-F1 (3 classes) : {macro_f1:.4f}')
    print('Per-Class Metrics (3 true classes):')
    for cls in three_classes:
        p = rep[cls]['precision']
        r = rep[cls]['recall']
        f = rep[cls]['f1-score']
        s = rep[cls]['support']
        print(f'  {cls:15s}: Precision={p:.4f}, Recall={r:.4f}, F1={f:.4f} (support={s})')
    print('Predictions in Impossible Classes:')
    print(f'  hvac              : {n_hvac}')
    print(f'  music_loudspeaker : {n_music}')
    print(f'  total impossible  : {n_impossible} / 200 ({n_impossible/200*100:.1f}%)\n')

print('Cell 3 completed ✓')

=== SVM-clean on ESC-50 (Cross-Dataset, 200 clips) ===
Accuracy (200 clips) : 0.6350 (63.50%)
Macro-F1 (3 classes) : 0.7404
Per-Class Metrics (3 true classes):
  construction   : Precision=0.8421, Recall=0.6000, F1=0.7007 (support=80.0)
  siren          : Precision=0.8846, Recall=0.5750, F1=0.6970 (support=40.0)
  traffic_horn   : Precision=1.0000, Recall=0.7000, F1=0.8235 (support=80.0)
Predictions in Impossible Classes:
  hvac              : 12
  music_loudspeaker : 49
  total impossible  : 61 / 200 (30.5%)



=== RF-clean on ESC-50 (Cross-Dataset, 200 clips) ===
Accuracy (200 clips) : 0.6500 (65.00%)
Macro-F1 (3 classes) : 0.7628
Per-Class Metrics (3 true classes):
  construction   : Precision=0.8596, Recall=0.6125, F1=0.7153 (support=80.0)
  siren          : Precision=0.9630, Recall=0.6500, F1=0.7761 (support=40.0)
  traffic_horn   : Precision=0.9483, Recall=0.6875, F1=0.7971 (support=80.0)
Predictions in Impossible Classes:
  hvac              : 5
  music_loudspeaker : 53
  total impossible  : 58 / 200 (29.0%)



=== SVM-mixed on ESC-50 (Cross-Dataset, 200 clips) ===
Accuracy (200 clips) : 0.6350 (63.50%)
Macro-F1 (3 classes) : 0.7300
Per-Class Metrics (3 true classes):
  construction   : Precision=0.8519, Recall=0.5750, F1=0.6866 (support=80.0)
  siren          : Precision=0.8462, Recall=0.5500, F1=0.6667 (support=40.0)
  traffic_horn   : Precision=0.9672, Recall=0.7375, F1=0.8369 (support=80.0)
Predictions in Impossible Classes:
  hvac              : 6
  music_loudspeaker : 53
  total impossible  : 59 / 200 (29.5%)



=== RF-mixed on ESC-50 (Cross-Dataset, 200 clips) ===
Accuracy (200 clips) : 0.6000 (60.00%)
Macro-F1 (3 classes) : 0.7129
Per-Class Metrics (3 true classes):
  construction   : Precision=0.8214, Recall=0.5750, F1=0.6765 (support=80.0)
  siren          : Precision=0.8333, Recall=0.6250, F1=0.7143 (support=40.0)
  traffic_horn   : Precision=0.9608, Recall=0.6125, F1=0.7481 (support=80.0)
Predictions in Impossible Classes:
  hvac              : 5
  music_loudspeaker : 58
  total impossible  : 63 / 200 (31.5%)

Cell 3 completed ✓


In [4]:
# Cell 4 – confusion matrix (rows: 3 true classes, cols: all 5 predicted classes) & save SVM-clean figure
row_classes = ['construction', 'siren', 'traffic_horn']
col_classes = ['construction', 'hvac', 'music_loudspeaker', 'siren', 'traffic_horn']

cms_3x5 = {}

for model_name, res in cross_eval.items():
    y_p = res['y_pred']
    cm = np.zeros((len(row_classes), len(col_classes)), dtype=int)
    for i, true_c in enumerate(row_classes):
        for j, pred_c in enumerate(col_classes):
            cm[i, j] = int(np.sum((y_esc == true_c) & (y_p == pred_c)))
    cms_3x5[model_name] = cm
    
    cm_df = pd.DataFrame(cm, index=row_classes, columns=col_classes)
    cm_df.index.name = 'True'
    cm_df.columns.name = 'Predicted'
    print(f'=== {model_name} Cross-Dataset Confusion Matrix (3 true x 5 predicted) ===')
    print(cm_df.to_string())
    print()

# Save SVM-clean confusion matrix plot
fig_dir = REPO_ROOT / 'reports/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
svm_fig_path = fig_dir / 'svm_cross_dataset_confusion.png'

cm_svm_clean = cms_3x5['SVM-clean']
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(cm_svm_clean, cmap='Blues', aspect='auto')

ax.set_xticks(np.arange(len(col_classes)))
ax.set_yticks(np.arange(len(row_classes)))
ax.set_xticklabels(col_classes, fontsize=10, rotation=25, ha='right')
ax.set_yticklabels(row_classes, fontsize=10)
ax.set_xlabel('Predicted Class', fontsize=11, fontweight='bold')
ax.set_ylabel('True Class (ESC-50)', fontsize=11, fontweight='bold')
ax.set_title('SVM-clean Cross-Dataset Confusion Matrix (ESC-50)', fontsize=12, fontweight='bold')

thresh = cm_svm_clean.max() / 2.0
for i in range(len(row_classes)):
    for j in range(len(col_classes)):
        val = cm_svm_clean[i, j]
        color = 'white' if val > thresh else 'black'
        ax.text(j, i, str(val), ha='center', va='center', color=color, fontsize=11, fontweight='bold')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(svm_fig_path, dpi=150)
plt.close(fig)

assert svm_fig_path.exists(), f'Figure file not found: {svm_fig_path}'
print(f'Saved SVM-clean cross-dataset confusion matrix plot to: {svm_fig_path}')
print('Cell 4 completed ✓')

=== SVM-clean Cross-Dataset Confusion Matrix (3 true x 5 predicted) ===
Predicted     construction  hvac  music_loudspeaker  siren  traffic_horn
True                                                                    
construction            48     5                 24      3             0
siren                    2     0                 15     23             0
traffic_horn             7     7                 10      0            56

=== RF-clean Cross-Dataset Confusion Matrix (3 true x 5 predicted) ===
Predicted     construction  hvac  music_loudspeaker  siren  traffic_horn
True                                                                    
construction            49     1                 26      1             3
siren                    0     0                 14     26             0
traffic_horn             8     4                 13      0            55

=== SVM-mixed Cross-Dataset Confusion Matrix (3 true x 5 predicted) ===
Predicted     construction  hvac  music_loudspeaker  

Saved SVM-clean cross-dataset confusion matrix plot to: C:\Users\LENOVO\Desktop\CityPulse\reports\figures\svm_cross_dataset_confusion.png
Cell 4 completed ✓


In [5]:
# Cell 5 – fair in-dataset comparison (fold 10 restricted to the same 3 classes, 405 clips)
X_test_clean = audio_features[test_idx]
y_test_clean = labels.loc[test_idx, 'app_class'].to_numpy()

three_classes = ['construction', 'siren', 'traffic_horn']
mask_fold10_3cls = np.isin(y_test_clean, three_classes)

X_test_3cls = X_test_clean[mask_fold10_3cls]
y_test_3cls = y_test_clean[mask_fold10_3cls]

print(f'Total fold-10 clips               : {len(y_test_clean)}')
print(f'Fold-10 3-class clips             : {len(y_test_3cls)}')
for cls in three_classes:
    print(f'  {cls:15s}: {int(np.sum(y_test_3cls == cls))}')

assert len(y_test_3cls) == 405, f'Expected exactly 405 clips in fold 10 for 3 classes, got {len(y_test_3cls)}'

comparison_rows = []
for model_name, model in models.items():
    y_pred_fold10_3 = model.predict(X_test_3cls)
    in_acc = float(accuracy_score(y_test_3cls, y_pred_fold10_3))
    cross_acc = cross_eval[model_name]['accuracy']
    drop_pts = (in_acc - cross_acc) * 100.0
    
    comparison_rows.append({
        'model': model_name,
        'in_dataset_3cls_acc': in_acc,
        'cross_dataset_acc': cross_acc,
        'drop_points': drop_pts
    })

df_comparison = pd.DataFrame(comparison_rows)
print('\n=== In-Dataset (Fold 10, 3 classes) vs Cross-Dataset (ESC-50) ===')
print(df_comparison.to_string(index=False, formatters={
    'in_dataset_3cls_acc': '{:.4f}'.format,
    'cross_dataset_acc': '{:.4f}'.format,
    'drop_points': '{:+.2f}'.format
}))
print('\nCell 5 completed ✓')

Total fold-10 clips               : 605
Fold-10 3-class clips             : 405
  construction   : 196
  siren          : 83
  traffic_horn   : 126



=== In-Dataset (Fold 10, 3 classes) vs Cross-Dataset (ESC-50) ===
    model in_dataset_3cls_acc cross_dataset_acc drop_points
SVM-clean              0.7901            0.6350      +15.51
 RF-clean              0.7383            0.6500       +8.83
SVM-mixed              0.7704            0.6350      +13.54
 RF-mixed              0.7630            0.6000      +16.30

Cell 5 completed ✓


In [6]:
# Cell 6 – bootstrap 95% confidence intervals (1000 resamples, seed=42)
N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
boot_indices = rng.choice(len(y_esc), size=(N_BOOTSTRAP, len(y_esc)), replace=True)

bootstrap_results = {}
print('=== Bootstrap 95% Confidence Intervals for Cross-Dataset Accuracy (1000 resamples, seed=42) ===')
for model_name in models.keys():
    y_pred = cross_eval[model_name]['y_pred']
    correct = (y_pred == y_esc).astype(float)
    
    boot_accs = np.array([np.mean(correct[idx]) for idx in boot_indices])
    ci_lower = float(np.percentile(boot_accs, 2.5))
    ci_upper = float(np.percentile(boot_accs, 97.5))
    bootstrap_results[model_name] = {
        'ci_95_lower': ci_lower,
        'ci_95_upper': ci_upper
    }
    
    acc_orig = cross_eval[model_name]['accuracy']
    print(f'{model_name:12s}: Cross-dataset Acc = {acc_orig:.4f} ({acc_orig*100:.2f}%), 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}] ([{ci_lower*100:.2f}%, {ci_upper*100:.2f}%])')

print('\nCell 6 completed ✓')

=== Bootstrap 95% Confidence Intervals for Cross-Dataset Accuracy (1000 resamples, seed=42) ===
SVM-clean   : Cross-dataset Acc = 0.6350 (63.50%), 95% CI: [0.5700, 0.7050] ([57.00%, 70.50%])
RF-clean    : Cross-dataset Acc = 0.6500 (65.00%), 95% CI: [0.5849, 0.7150] ([58.49%, 71.50%])
SVM-mixed   : Cross-dataset Acc = 0.6350 (63.50%), 95% CI: [0.5650, 0.6950] ([56.50%, 69.50%])
RF-mixed    : Cross-dataset Acc = 0.6000 (60.00%), 95% CI: [0.5350, 0.6650] ([53.50%, 66.50%])

Cell 6 completed ✓


In [7]:
# Cell 7 – save reports/cross_dataset_eval.csv and print sha256
export_rows = []
for model_name in models.keys():
    comp = next(r for r in comparison_rows if r['model'] == model_name)
    ceval = cross_eval[model_name]
    bstrap = bootstrap_results[model_name]
    
    export_rows.append({
        'model': model_name,
        'in_dataset_3cls_acc': comp['in_dataset_3cls_acc'],
        'cross_dataset_acc': comp['cross_dataset_acc'],
        'drop_points': comp['drop_points'],
        'ci_95_lower': bstrap['ci_95_lower'],
        'ci_95_upper': bstrap['ci_95_upper'],
        'macro_f1': ceval['macro_f1'],
        'f1_construction': ceval['per_class']['construction']['f1-score'],
        'f1_siren': ceval['per_class']['siren']['f1-score'],
        'f1_traffic_horn': ceval['per_class']['traffic_horn']['f1-score'],
        'n_impossible': ceval['n_impossible'],
        'n_hvac': ceval['n_hvac'],
        'n_music_loudspeaker': ceval['n_music_loudspeaker']
    })

df_export = pd.DataFrame(export_rows)
reports_dir = REPO_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
csv_path = reports_dir / 'cross_dataset_eval.csv'
df_export.to_csv(csv_path, index=False)

assert csv_path.exists(), f'File not found: {csv_path}'
assert len(df_export) == 4, f'Expected 4 rows, got {len(df_export)}'

print(f'Saved {csv_path}')
print('\n=== Exported reports/cross_dataset_eval.csv ===')
print(df_export.round(4).to_string(index=False))

csv_sha = compute_sha256(csv_path)
fig_sha = compute_sha256(svm_fig_path)

print('\n=== SHA256 Hashes of New Files ===')
print(f'reports/cross_dataset_eval.csv               : {csv_sha}')
print(f'reports/figures/svm_cross_dataset_confusion.png: {fig_sha}')
print('\nCell 7 completed ✓')

Saved C:\Users\LENOVO\Desktop\CityPulse\reports\cross_dataset_eval.csv

=== Exported reports/cross_dataset_eval.csv ===
    model  in_dataset_3cls_acc  cross_dataset_acc  drop_points  ci_95_lower  ci_95_upper  macro_f1  f1_construction  f1_siren  f1_traffic_horn  n_impossible  n_hvac  n_music_loudspeaker
SVM-clean               0.7901              0.635      15.5123       0.5700        0.705    0.7404           0.7007    0.6970           0.8235            61      12                   49
 RF-clean               0.7383              0.650       8.8272       0.5849        0.715    0.7628           0.7153    0.7761           0.7971            58       5                   53
SVM-mixed               0.7704              0.635      13.5370       0.5650        0.695    0.7300           0.6866    0.6667           0.8369            59       6                   53
 RF-mixed               0.7630              0.600      16.2963       0.5350        0.665    0.7129           0.6765    0.7143           